# DhakaFlow P2 — Sensor-Graph Generation (Drive-bundle variant)

Same as `P2_sensor_graph_colab.ipynb`, but sources the code from
`MyDrive/dhakaflow/dhakaflow_repo_bundle.zip` (repo @ HEAD + the frozen R1
network) instead of cloning GitHub — the repo is private and no GitHub
authorisation is wired into Colab. CPU runtime, no API keys.

Every output is `DHAKA_SIMULATED_SENSOR_GRAPH` / SIMULATED.

In [ ]:
# 1 · SUMO via pip + Drive mount + unpack the bundle.
%pip -q install eclipse-sumo sumolib numpy
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/dhakaflow/artifacts/p2_sensor_graph'

import glob, os, subprocess, sys, zipfile
# Bundle may be one zip or split bundle_part_* files, in MyDrive root or dhakaflow/.
roots = ['/content/drive/MyDrive/dhakaflow', '/content/drive/MyDrive']
BUNDLE = next((p for r in roots
               for p in [os.path.join(r, 'dhakaflow_repo_bundle.zip')]
               if os.path.exists(p)), None)
if BUNDLE is None:
    for r in roots:
        parts = sorted(glob.glob(os.path.join(r, 'bundle_part_*')))
        if parts:
            BUNDLE = '/content/dhakaflow_repo_bundle.zip'
            with open(BUNDLE, 'wb') as out_f:
                for part in parts:
                    out_f.write(open(part, 'rb').read())
            break
assert BUNDLE, 'bundle not found on Drive'
REPO_DIR = '/content/thesis'
if not os.path.exists(REPO_DIR):
    os.makedirs(REPO_DIR)
    with zipfile.ZipFile(BUNDLE) as z:
        z.extractall(REPO_DIR)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(subprocess.run(['sumo','--version'], capture_output=True, text=True).stdout.splitlines()[0])
# The frozen-network checksum is verified by SumoTwin before any run.
from dhakaflow.common.city import DHAKA_R1
ok, actual = DHAKA_R1.verify_network()
assert ok, f'frozen network checksum mismatch: {actual}'
print('frozen R1 network verified:', actual[:16], '…')

In [ ]:
# 2 · Smoke first (repo rule) — ~2 min. Must finish with insertion 1.00.
!python scripts/run_sensor_graph.py --mode smoke --out /content/p2_smoke

In [ ]:
# 3 · FULL generation: 24 h × 3 seeds, straight to Drive. Budget hours of
#     CPU: full-scale peaks jam and SUMO slows there — that is the point.
!python scripts/run_sensor_graph.py --mode full --seeds 42 43 44 --out {DRIVE_OUT}

In [ ]:
# 4 · Acceptance checks + PASS evidence on Drive (P2 is not PASS without these).
import json, numpy as np, hashlib, pathlib
out = pathlib.Path(DRIVE_OUT)
m = json.loads((out/'MANIFEST.json').read_text())
z = np.load(out/'sensor_graph_data.npz')
data, mask, adj = z['data'], z['mask'], z['adjacency']
checks = {
  'evidence_domain_is_simulated': m['evidence_domain'] == 'DHAKA_SIMULATED_SENSOR_GRAPH',
  'nodes_in_mandate_100_300': 100 <= data.shape[1] <= 300,
  'three_days_of_5min_bins': data.shape[0] == 3*24*12,
  'four_channels': data.shape[2] == 4,
  'adjacency_square': adj.shape == (data.shape[1], data.shape[1]),
  'peak_bins_well_covered': float(mask[(np.arange(data.shape[0])%288)//12==8].mean()) > 0.5,
  'speed_nan_only_where_masked': bool(np.isnan(data[...,0])[mask].sum() == 0),
}
print(json.dumps(checks, indent=1))
assert all(checks.values()), 'P2 acceptance checks FAILED — do not mark PASS'
sha = hashlib.sha256((out/'sensor_graph_data.npz').read_bytes()).hexdigest()
(out/'P2_PASS_EVIDENCE.json').write_text(json.dumps({
  'checks': checks, 'data_sha256': sha,
  'manifest_node_hash': m['node_list_sha256']}, indent=2))
print('P2 acceptance checks all green — evidence on Drive:', out/'P2_PASS_EVIDENCE.json')

In [ ]:
# 5 · Bonus while the session is warm: P3 floors on the full dataset → Drive.
!python scripts/run_forecast_baselines.py --data {DRIVE_OUT}
print('floors written next to the dataset on Drive')